# Домашнее задание 5

QLoRA-дообучение языковой модели для задачи генерации текста + профайлинг обучения.

## 0. Установка зависимсотей

In [1]:
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install trl datasets accelerate peft bitsandbytes transformers sentencepiece lm-eval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 21.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 3.5 MB/

In [2]:
import os, gc, math, time, json, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import TrainingArguments

from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

/tmp/ipykernel_381/2011144717.py:11: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth.chat_templates import get_chat_template


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
import warnings
warnings.simplefilter('ignore')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def cuda_mem(label=''):
    if not torch.cuda.is_available():
        print('CUDA not available'); return
    torch.cuda.synchronize()
    a = torch.cuda.memory_allocated() / 1024**2
    r = torch.cuda.memory_reserved() / 1024**2
    m = torch.cuda.max_memory_allocated() / 1024**2
    print(f'[VRAM] {label} allocated={a:.0f} MB | reserved={r:.0f} MB | max_alloc={m:.0f} MB')

def reset_cuda_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

In [4]:
SEED = 42
MAX_SEQ_LENGTH = 1024
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit' # обоснование ниже
DATASET_NAME = 'serguntsov/medicalQA' # обоснование ниже

# корзина промптов для сравнения до/после
prompts_for_test = [
    'У меня уже неделю болит горло и температура 37.5. Что делать?',
    'Ребёнок (5 лет) пожаловался на боль в животе справа внизу, есть тошнота. К какому врачу идти?',
    'Можно ли пить парацетамол и ибупрофен одновременно?',
    'Давление 150/95, пульс 88, чувствую головокружение. Это опасно?',
    'Какой врач занимается лечением хронического гастрита и какие анализы обычно назначают?',
]

set_seed(SEED)
cuda_mem('start')

[VRAM] start allocated=8 MB | reserved=22 MB | max_alloc=8 MB


## 1. Выбор и загрузка датасета

Обучаться будем в режиме SFT, тк это основной путь адаптации LLM под конкретный домен и сценарий, а также дает видимый эффект на малых данных, в отличие от CPT.

В первой итерации пробовал датасет `blinoff/medical_qa_ru_data` (190к, нативный русский, форумные Q&A на 10 специальностей), но в нем оказалось слишком много форумных артефактов, которые не получилось почистить.

Поэтому выбор пал на датасет `serguntsov/medicalQA` с 70к+ чистых Q&A пар. Относительно `blinoff/medical_qa_ru_data` теряем нативный русский (иногда проскакивают кальки с английского), но взамен получаем чистый формат под SFT задачу и развернутые ответы

Возьмём случайную подвыборку из 5к записей - этого должно быть достаточно

In [5]:
from huggingface_hub import hf_hub_download

pq_path = hf_hub_download(repo_id=DATASET_NAME, filename='data/train-00000-of-00001.parquet', repo_type='dataset')
df_raw = pd.read_parquet(pq_path)
df_raw.shape

data/train-00000-of-00001.parquet:   0%|          | 0.00/76.5M [00:00<?, ?B/s]

(71170, 2)

In [6]:
df = df_raw.copy()
print('shape:', df.shape)
print('columns:', df.columns.tolist())
print()
print(pd.DataFrame({'question': df['question'].str.len(), 'answer': df['answer'].str.len()}).describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0))
print()
print('Q:', df.iloc[0]['question'][:200])
print('A:', df.iloc[0]['answer'][:300])

shape: (71170, 2)
columns: ['question', 'answer']

       question   answer
count   71170.0  71170.0
mean      664.0    583.0
std       582.0    431.0
min         0.0      3.0
50%       567.0    459.0
90%      1518.0   1189.0
95%      1828.0   1544.0
99%      2228.0   2002.0
max      3536.0   4183.0

Q: Что вызывает опухоли центральной нервной системы у взрослых?
A: Причина большинства опухолей головного и спинного мозга у взрослых неизвестна.


In [7]:
# почистим дубли, слишком короткие (справочные заглушки без текста) и слишком длинные (не влезут в max_seq) ответы, пропуски
df = df.dropna(subset=['question', 'answer']).copy()
df['question'] = df['question'].astype(str).str.strip()
df['answer']   = df['answer'].astype(str).str.strip()
df = df[(df['question'].str.len() >= 20) & (df['answer'].str.len() >= 100)]
df = df[df['answer'].str.len() <= 3000]
df = df.drop_duplicates(subset=['question', 'answer']).reset_index(drop=True)
df.shape

(67797, 2)

In [8]:
# случайная подвыборка из 5к записей
sample = df.sample(n=5000, random_state=SEED).reset_index(drop=True)
sample.shape

(5000, 2)

In [9]:
# train/val 90/10
train_df, val_df = train_test_split(sample, test_size=0.1, random_state=SEED)

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['question', 'answer']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['question', 'answer']].reset_index(drop=True)),
})
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 4500
    })
    validation: Dataset({
        features: ['question', 'answer'],
        num_rows: 500
    })
})

## 2. Выбор предобученной модели

Выбор пал на модель `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` по нескольким причинам:

- 1.5B - верх диапазона по условия, при этом в 4-bit она спокойно влезает в T4 на collab
- Сильный русский - Qwen2.5 в классе моделей <2B обгоняет конкурентов, благодаря тому, что был сделан фокус на мультиязычности.
- Instruct выбран потому, что хорошо нативно работает с чатовым форматом (то что и надо для нашего датасета)

In [10]:
reset_cuda_peak()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH, # покрывает p99 из наших данных, так что будет достаточно
    dtype=None,
    load_in_4bit=True,
)

cuda_mem('after model load (4bit)')

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[VRAM] after model load (4bit) allocated=1134 MB | reserved=1160 MB | max_alloc=1150 MB


## 3. Предварительная оценка качества

### 3.1 Корзинка тестовых примеров

In [11]:
# Переключем модель на inference mode для ускорения
FastLanguageModel.for_inference(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RM

In [ ]:
# соберем каркас для тестирования модели до/после тюнинга
# параметры модели выставлены как стандарт (будем сравнивать на одном наборе параметров)
SYSTEM_PROMPT = 'Ты - полезный медицинский ассистент. Давай краткие, понятные ответы и при необходимости рекомендуй обратиться к врачу.'

def generate_chat(model, tokenizer, user_text, system_text=SYSTEM_PROMPT, max_new_tokens=256):
    messages = [
        {'role': 'system', 'content': system_text},
        {'role': 'user', 'content': user_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [13]:
# посмотрим глазами, как справится модель "из коробки"
for p in prompts_for_test:
    print(p)
    print('-'*80)
    print(generate_chat(model, tokenizer, p)[:1500])
    print('='*80)

У меня уже неделю болит горло и температура 37.5. Что делать?
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Вы посетили стационар? Если нет, лучше обратиться к врачу для обследования и получения консультации. Врач может прописать лекарства или назначить другие методы лечения. Если есть возможность, уделите внимание своим здоровью. Если температура не снизится в течение дня, возможно, нужно обратиться к врачу для более тщательного обследования.
Ребёнок (5 лет) пожаловался на боль в животе справа внизу, есть тошнота. К какому врачу идти?
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Если у вашего ребенка появляется тошнота и боль в животе справа внизу, это может быть связано с различными причинами, такими как глистозаконченное пищевое воспаление, инфекционные заболевания или непереносимые продукты питания.

Ваш ребенок должен быть отправлен к специалисту для консультации. Если возможно, лучше сразу же посетить педиатра или молекулярно-ферментологию, чтобы исключить серьезные проблемы. В случае отсутствия определенных симптомов или если они усиливаются, необходимо проконсультироваться со стоматологом или эндокринологом, так как эти проблемы могут также иметь место. Не забывайте о безопасности — всегда можно посетить семейный доктор, который сможет оценить ситуацию и предложить необходимую помощь.
Можно ли пить парацетамол и ибупрофен одновременно?
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Парацетамол и ибупрофен оба являются антиинflammatory-аналитические препараты, но они имеют свои особенности применения:

1. Ибупрофен - более интенсивно влияет на боли в спине и грудной клетчатой полости.
2. Парацетамол - эффективнее в отношении головных болей и сильных мигрени.

В некоторых случаях может быть полезна комбинация этих препаратов для уменьшения болевых ощущений, особенно в тяжелых или постоянных формах заболевания.

Однако следует помнить:
- Симптомы могут усиливаться при совместном применении;
- Вы должны проконсультироваться со специалистом перед началом лечения;
- Необходимо отслеживать возможные побочные эффекты и общую реакцию организма на комбинацию препаратов.

Если вы принимаете какие-либо другие лекарства, всегда лучше получить консультацию с врачом прежде чем начать новое лечение.
Давление 150/95, пульс 88, чувствую головокружение. Это опасно?
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Да, это может быть опасно. Вам нужно проконсультироваться с врачом для оценки состояния. Головокружение вместе с повышенным давлением (плюс низкий пульс) могут указывать на сердечную недостаточность или инфекцию в горле, что требует немедленного медицинского обследования. Если у вас есть тревожные симптомы, не откладывайте обращение к доктору.
Какой врач занимается лечением хронического гастрита и какие анализы обычно назначают?
--------------------------------------------------------------------------------
Хронический гастрит лечится специалистами-гастроэнтерологами. Для диагностики и лечения часто назначаются такие анализы:

1. Эпигастрический салон (диагностика патогенетической активности)
2. Глюкозохромоанализ (для выявления метаболических нарушений)
3. Печеночные функции (тестирование для определения общей концентрации белков в печени)
4. Ферроцитопластидоанализ (для оценки ферроцитоплазматической активности)

Эти анализы помогают установить причины заболевания и подобрать эффект

### Что видно "из коробки"?

- есть фундамент для работы с русским: текст связный, лексика почти всегда валидная, есть структура
- много галлюцинаций медицинской терминологии: "молекулярно-ферментология", "эпигастрический салон", "ферроцитопластидоанализ" и тд
- есть странные вкрапления английского прямо в слова: "антиинflammatory", "этиsymptom" и тд
- есть фактические ошибки: пульс 88 назван низким, хотя это может быть нормой; не распознал классические симптомы аппендицита; при температуре 37.5 направляет в стационар

P.S. выводы в ячейке выше обновлялись, но все проблемы прослеживаются явно

### 3.2 Оценка на lm-evaluation-harness

Для оценки выбрал 2 задачи:
- `truthfulqa_mc2` - стандартный бенчмарк общего качества LLM (как делали в семинаре)
- `mmlu_clinical_knowledge` - бенчмарк по медицинскому домену как раз в формате SFT: единственное узкое горлышко - английский язык, но на русском готового бенчмарка найти не удалось

In [14]:
# освобождаем память после инференса (иначе создадим новый инстант)
del model
gc.collect()
torch.cuda.empty_cache()
reset_cuda_peak()
cuda_mem('before lm_eval')

[VRAM] before lm_eval allocated=1152 MB | reserved=1184 MB | max_alloc=1152 MB


In [15]:
BASE_BENCH_DIR = '/content/bench_base'
os.makedirs(BASE_BENCH_DIR, exist_ok=True)

In [16]:
!lm_eval \
  --model hf \
  --model_args pretrained={MODEL_NAME},trust_remote_code=True \
  --tasks truthfulqa_mc2,mmlu_clinical_knowledge \
  --device cuda:0 \
  --batch_size 4 \
  --limit 50 \
  --output_path {BASE_BENCH_DIR}

2026-04-22:12:48:34 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-22:12:48:41 INFO     [_cli.run:376] Selected Tasks: ['truthfulqa_mc2', 'mmlu_clinical_knowledge']
2026-04-22:12:48:41 WARNING  [evaluator:181] pretrained=unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit appears to be an instruct or chat variant but chat template is not applied. Recommend
        setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-04-22:12:48:41 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-04-22:12:48:41 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit', 'trust_remote_code': True}
2026-04-22:12:48:46 INFO     [models.huggingface:161] Using device 'cuda:0'
tokenizer_config.json: 7.36kB [00:00, 8.57MB/s]
vocab.json: 2.78MB [00:00

## 4. QLoRA

### 4.1 Конфигурация квантования (bitsandbytes 4-bit)

In [17]:
# чистим память и грузим модель в 4-bit
gc.collect(); torch.cuda.empty_cache(); reset_cuda_peak()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

cuda_mem('after reload (4bit)')

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[VRAM] after reload (4bit) allocated=2275 MB | reserved=2296 MB | max_alloc=2291 MB


### 4.2 Конфигурация LoRA

In [18]:
# навешиваем LoRA на все линейные слои (attention + MLP)
def add_lora(model, r=16, lora_alpha=32, dropout=0.0):
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
    model = FastLanguageModel.get_peft_model(
        model,
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=dropout,
        bias='none',
        target_modules=target_modules,
        use_gradient_checkpointing=True, # trade-off скорости на экономию памяти на colab
        random_state=SEED,
        use_rslora=False,
    )
    return model

def trainable_params_report(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {'trainable': trainable, 'total': total, 'trainable_pct': round(100*trainable/total, 3)}

model = add_lora(model, r=16, lora_alpha=32, dropout=0.0)
trainable_params_report(model)

Unsloth 2026.4.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


{'trainable': 18464768, 'total': 907081216, 'trainable_pct': 2.036}

### 4.3 Подготовка данных для SFT

In [ ]:
# собираем пары (question, answer) в chat_template модели - так готовим текст для SFT
def to_text(example):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': example['question']},
        {'role': 'assistant', 'content': example['answer']},
    ]
    # add_generation_prompt=False — ответ уже в messages, финальный <|im_start|>assistant не нужен
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    n_tokens = len(tokenizer(text, add_special_tokens=False).input_ids)
    return {'text': text, 'n_tokens': n_tokens}

ds_train = dataset['train'].map(to_text, remove_columns=['question', 'answer'])
ds_val   = dataset['validation'].map(to_text, remove_columns=['question', 'answer'])

# дроп длинных пар (чтобы не сломать структуру ответов модели и не обрывать их)
before = (len(ds_train), len(ds_val))
ds_train = ds_train.filter(lambda x: x['n_tokens'] <= MAX_SEQ_LENGTH)
ds_val   = ds_val.filter(lambda x: x['n_tokens'] <= MAX_SEQ_LENGTH)
print(f'dropped train: {before[0] - len(ds_train)} / {before[0]}')
print(f'dropped val:   {before[1] - len(ds_val)} / {before[1]}')
print('sample:\n', ds_train[0]['text'][:500])

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

dropped train: 115 / 4500
dropped val:   10 / 500
sample:
 <|im_start|>system
Ты — полезный медицинский ассистент. Давай краткие, понятные ответы и при необходимости рекомендуй обратиться к врачу.<|im_end|>
<|im_start|>user
20-летний мужчина
Рост 175 см
Вес 76 кг
Худощавое телосложение

Когда что-то вызывает у меня тревогу, заставляет вздрагивать или сильно беспокоиться, я чувствую, что мое сердце очень сильно бьется, люди, которые кладут руку мне на грудь, довольно шокированы. Но у меня было несколько ЭКГ, и значительных аритмий не выявлено. Может ли т


### 4.4 Обучение

In [20]:
sft_args = SFTConfig(
    output_dir='/content/sft_run',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # эффективный batch = 2*4 = 8, больше на colab с max_seq=1024 не влезет
    learning_rate=2e-4,             # стандарт для LoRA
    warmup_ratio=0.03,              # небольшой прогрев
    lr_scheduler_type='cosine',     # к концу наоборот смягчаем
    max_steps=300,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='no',
    optim='paged_adamw_8bit',
    weight_decay=0.0,
    fp16=not torch.cuda.is_bf16_supported(), # T4 на colab не умеет работать с bf16 — автоматом в fp16
    bf16=torch.cuda.is_bf16_supported(),
    report_to='none',
    seed=SEED,
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    args=sft_args,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4385 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/4385 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/490 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=6):   0%|          | 0/490 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [21]:
# запускаем обучение и меряем время + пик нагрузки по памяти
cuda_mem('before train')
t0 = time.time()
train_out = trainer.train()
t1 = time.time()
cuda_mem('after train')
print(f'train time: {(t1-t0)/60:.1f} min')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


[VRAM] before train allocated=2344 MB | reserved=2370 MB | max_alloc=2344 MB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,003 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,1.500786,1.471161
100,1.438811,1.411833
150,1.406093,1.387639
200,1.367858,1.372187
250,1.388643,1.366053
300,1.331673,1.365284


[VRAM] after train allocated=2394 MB | reserved=3322 MB | max_alloc=4183 MB
train time: 72.1 min


## 5. Оценка качества после обучения

In [22]:
# те же 5 промптов, тот же sys prompt, тот же generate_chat для честного сравнения
FastLanguageModel.for_inference(trainer.model)

for p in prompts_for_test:
    print(p)
    print('-'*80)
    print(generate_chat(trainer.model, tokenizer, p)[:1500])
    print('='*80)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


У меня уже неделю болит горло и температура 37.5. Что делать?
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Симптомы ухудшаются или нет?

Информация носит общий ознакомительный характер. Для диагностики и лечения необходима консультация врача.
Ребёнок (5 лет) пожаловался на боль в животе справа внизу, есть тошнота. К какому врачу идти?
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Симптомы могут указывать на инфекцию мочевыводящих путей или диарею, но для точной диагностики и назначения лечения необходима консультация педиатра или儿科хирурга. Информация носит общий ознакомительный характер; для диагностики и лечения необходима обратка к специалистам.
Можно ли пить парацетамол и ибупрофен одновременно?
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Парацетамол и ибупрофен могут использоваться для лечения симптомов неврологической недостаточности (например, головокружения), но не являются первым способом лечения.

Если вы принимаете любое лекарство по назначению врача, важно соблюдать инструкции относительно приема дозировки и интервалов между приемами. Если у вас есть вопросы о приеме этих препаратов или других лекарств, связанных с вашей неврологической недостаточностью, необходима консультация вашего лечащего врача.
Давление 150/95, пульс 88, чувствую головокружение. Это опасно?
--------------------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Пожалуйста, не связывайте это с вашим весом. Следует отметить, что информация носит общий ознакомительный характер. Для диагностики и лечения необходима консультация врача.
Какой врач занимается лечением хронического гастрита и какие анализы обычно назначают?
--------------------------------------------------------------------------------
Хронический гастрит чаще всего диагностируется по симптомам. Для точной диагностики необходима консультация врача-гastroenterолога.

Для подтверждения диагноза и определения причин гастрита может потребоваться:
Эндоскопическое исследование (также называемое эзофагогастроскопическое исследование)
Колоноскопия

Анализ крови (антиген КТГ)


In [ ]:
# сохраняем lora адаптер отдельно - для lm_eval subprocess и на случай повторного использования
ADAPTER_DIR = '/content/artifacts/adapter'
os.makedirs(ADAPTER_DIR, exist_ok=True)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

!du -sh {ADAPTER_DIR}

82M	/content/artifacts/adapter


In [ ]:
# освобождаем память под subprocess lm_eval
del trainer, model
gc.collect()
torch.cuda.empty_cache()
reset_cuda_peak()
cuda_mem('before lm_eval (sft)')

[VRAM] before lm_eval (sft) allocated=2436 MB | reserved=2548 MB | max_alloc=2436 MB


In [ ]:
SFT_BENCH_DIR = '/content/bench_sft'
os.makedirs(SFT_BENCH_DIR, exist_ok=True)

In [ ]:
# те же tasks/limit/batch_size что и в baseline
# peft=ADAPTER_DIR - lm_eval подхватит и применит lora адаптер поверх базовой модели
!lm_eval \
  --model hf \
  --model_args pretrained={MODEL_NAME},peft={ADAPTER_DIR},trust_remote_code=True \
  --tasks truthfulqa_mc2,mmlu_clinical_knowledge \
  --device cuda:0 \
  --batch_size 4 \
  --limit 50 \
  --output_path {SFT_BENCH_DIR}

2026-04-22:11:19:14 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-22:11:19:22 INFO     [_cli.run:376] Selected Tasks: ['truthfulqa_mc2', 'mmlu_clinical_knowledge']
2026-04-22:11:19:22 WARNING  [evaluator:181] pretrained=unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit appears to be an instruct or chat variant but chat template is not applied. Recommend
        setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-04-22:11:19:22 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-04-22:11:19:22 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit', 'peft': '/content/artifacts/adapter', 'trust_remote_code': True}
2026-04-22:11:19:27 INFO     [models.huggingface:161] Using device 'cuda:0'
Skipping import of cpp extensions d

In [ ]:
# собьем результаты бенчмарков до/после
import glob

def read_lm_eval_dir(d):
    files = sorted(glob.glob(f'{d}/**/*.json', recursive=True))
    assert files, f'no results in {d}'
    with open(files[-1]) as f:
        return json.load(f)['results']

base = read_lm_eval_dir(BASE_BENCH_DIR)
sft  = read_lm_eval_dir(SFT_BENCH_DIR)

rows = []
for task in ['truthfulqa_mc2', 'mmlu_clinical_knowledge']:
    metric_key = next((k for k in base[task] if k.startswith('acc,')), 'acc,none')
    stderr_key = metric_key.replace('acc', 'acc_stderr')
    b = base[task].get(metric_key, base[task].get('acc'))
    s = sft[task].get(metric_key, sft[task].get('acc'))
    b_err = base[task].get(stderr_key, 0)
    s_err = sft[task].get(stderr_key, 0)
    rows.append({'task': task, 'base': round(b, 3), 'sft': round(s, 3), 'delta': round(s-b, 3), 'base_stderr': round(b_err, 3), 'sft_stderr': round(s_err, 3)})

pd.DataFrame(rows)

,task,base,sft,delta,base_stderr,sft_stderr
0,truthfulqa_mc2,0.511,0.557,0.045,0.059,0.056
1,mmlu_clinical_knowledge,0.620,0.680,0.060,0.069,0.067


### Заключение по блоку 5

Что стало лучше относительно baseline:
- Ушли выдуманные термины («молекулярно-ферментология», «эпигастрический салон» и пр.) и переключение на английский
- Стиль стал более медицинский: появились уточняющие вопросы, стабильный дисклеймер в конце
- Ответы стали более конкретные и с деталями

Новые ошибки:
- Вкрапления других языков из переводного MedQuAD, наприме китайский иероглиф `儿科`
- Шаблонные фразы не всегда к месту: ответ про давление 150/95 про «не связывайте с вашим весом», хотя вес вообще не упоминался

Обе метрики чуть выросли, но при `limit=50` - статистически не значимо.

**Итого:** SFT сработал - модель начала говорить как медицинский ассистент, а не как универсальный чат-бот. Метрики не просели, но не выросли стат. значимо. Недостатки наследуются от плохого качества перевода датасета.

## 6. Профайлинг обучения

In [23]:
# снова грузим модель и навешиваем lora адаптер
gc.collect(); torch.cuda.empty_cache(); reset_cuda_peak()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
model = add_lora(model, r=16, lora_alpha=32, dropout=0.0) # та же конфигурация что в блоке 4
cuda_mem('after reload for profiling')

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[VRAM] after reload for profiling allocated=3642 MB | reserved=3670 MB | max_alloc=3642 MB


In [24]:
# построим профайлер процесса обучения
from torch.profiler import profile, ProfilerActivity, schedule, tensorboard_trace_handler
from transformers import TrainerCallback

PROF_DIR = '/content/prof_logs'
os.makedirs(PROF_DIR, exist_ok=True)

class ProfCallback(TrainerCallback):
    def __init__(self, prof):
        self.prof = prof
    def on_step_end(self, args, state, control, **kwargs):
        self.prof.step()

prof = profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=schedule(wait=1, warmup=1, active=3, repeat=1),
    on_trace_ready=tensorboard_trace_handler(PROF_DIR),
    record_shapes=True,
    profile_memory=True,
    with_stack=False,
)

In [25]:
# та же конфигурация обучения, но только 5 шагов (ровно wait + warmup + active)
prof_args = SFTConfig(
    output_dir='/content/sft_prof',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=5,   # wait=1 + warmup=1 + active=3
    logging_steps=1,
    save_strategy='no',
    eval_strategy='no',
    optim='paged_adamw_8bit',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to='none',
    seed=SEED,
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    packing=True,
)

trainer_prof = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds_train,
    args=prof_args,
    callbacks=[ProfCallback(prof)],
)

Unsloth: Sample packing skipped (custom data collator detected).


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/4385 [00:00<?, ? examples/s]

In [26]:
with prof:
    trainer_prof.train()

cuda_mem('after profiled train')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,385 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,2.012918
2,2.144054
3,2.044340
4,1.928509
5,1.909067


[VRAM] after profiled train allocated=3655 MB | reserved=7276 MB | max_alloc=4985 MB


In [27]:
# топ операций по CUDA-времени
print(prof.key_averages().table(sort_by='self_cuda_time_total', row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us       14.443s       104.40%       14.443s        4.814s           0 B           0 B           0 B           0 

In [28]:
# топ по CPU-времени
print(prof.key_averages().table(sort_by='self_cpu_time_total', row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*        31.34%        7.173s        63.21%       14.468s        4.823s       0.000us         0.00%        5.173s        1.724s           0 B           0 B     -45.00 KB     -15.21 G

In [29]:
# топ по памяти
print(prof.key_averages().table(sort_by='self_cuda_memory_usage', row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                            aten::empty         0.62%     142.716ms         0.62%     142.716ms       8.596us       0.000us         0.00%       0.000us       0.000us     451.25 KB     451.25 KB      58.94 GB      58.94 G

In [30]:
# путь до трейсов
!ls -la {PROF_DIR}

total 471060
drwxr-xr-x 2 root root      4096 Apr 22 14:11 .
drwxr-xr-x 1 root root      4096 Apr 22 14:10 ..
-rw-r--r-- 1 root root 482353442 Apr 22 14:11 8f1d08fd3d31_381.1776867062730671274.pt.trace.json


### Выводы по профайлингу

Потенциал для оптимизации есть:
- отключить gradient_checkpointing (+30% скорости, -2-3 ГБ памяти, может влезть на Т4 на colab)
- поднять batch_size до 4

Как таковых, узких мест в самом флоу нет - всё упирается в матричные умножения

## 7. Итоговые выводы

Дообучили `Qwen2.5-1.5B-Instruct` через QLoRA на медицинском датасете. Качественно SFT сработал: стиль общения стал более подходящий для медицинского ассиснтента. Обе метрики чуть подросли, но в пределах шума при `limit=50`.

P.S.Ресурса colab по времени не хватило, чтобы провести запуск на большей выборке и попробовать найти стат. значимый эффект, а времени на полный перезапуск с обучением не хватит =(

Главный урок - качество датасета важнее гиперпараметров. Первый заход на `blinoff` провалился из-за форумного шума в ответах, пришлось переключиться на чистый, но переведенный даатсет `serguntsov`. Баг датасета после перевода виден в финальных ответах: вкрапления других языков, шаблонные фразы не всегда к месту и тп.

Профайлинг показал такую картину: 72% CUDA-времени на matmul, 10% на дек-квантование nf4-fp16, CPU не стал bottleneck. На T4 по памяти оказался большой запас (пик 4.9 ГБ из 15) - можно было отключить gradient checkpointing или поднять batch, чтобы ускорить.